<a href="https://colab.research.google.com/github/i-wild/Python_Ejercicios/blob/master/CFDI-KPIengine-1200.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================================================
# KPI ENGINE V13.2 🔥
# SEPARA EMITIDAS/RECIBIDAS + CUADRE CORRECTO + ECE->ECE + AHORRO DE MEMORIA
# ============================================================

import pandas as pd
from google.colab import files
from IPython.display import Audio, display
import numpy as np
import csv

# ============================================================
# 🔔 CAMPANA
# ============================================================

def campana():
    duration = 0.5
    sr = 44100
    t = np.linspace(0, duration, int(sr * duration), False)
    tone = (0.6 * np.sin(2 * np.pi * 800 * t) + 0.3 * np.sin(2 * np.pi * 1200 * t))
    tone *= np.exp(-3 * t)
    display(Audio(tone, rate=sr, autoplay=True))

# ============================================================
# 🔧 FUNCIONES
# ============================================================

def safe_col(cols, keyword):
    for c in cols:
        if keyword.lower() in c.lower():
            return c
    raise ValueError(f"No existe columna {keyword}")

def clasificar_rfc(rfc):
    rfc = rfc.astype(str)
    return np.select(
        [
            rfc == "XAXX010101000",
            rfc == "XEXX010101000",
            rfc.str.len() == 12,
            rfc.str.len() == 13
        ],
        [
            "Genérico Nacional",
            "Extranjero",
            "Persona Moral",
            "Persona Física"
        ],
        default="Otro"
    )

def sanitize_sheet_name(name):
    invalid = ['\\', '/', '*', '?', ':', '[', ']']
    for ch in invalid:
        name = name.replace(ch, '-')
    return str(name)[:31]

def init_modo_data():
    return {
        "total_cfdi_cargados": 0,
        "total_cfdi_unicos": 0,
        "total_cfdi_analizados": 0,
        "total_monto_unicos": 0.0,
        "total_monto_analizados": 0.0,
        "duplicados": 0,
        "df_tipo_all": [],
        "df_rfc_all": [],
        "df_dia_all": [],
        "df_mes_cfdi_all": [],
        "df_mes_monto_all": [],
        "fechas_set": set(),
        "control_rows": [],
        "mensual_proveedor_rows": [],
        "ece_rows": [],
        "ece_mensual": []
    }

def procesar_chunk(chunk, uuid_set, ece_rfc, modos_data):
    chunk.columns = chunk.columns.str.strip()

    uuid_col = safe_col(chunk.columns, "Uuid")
    fecha_col = safe_col(chunk.columns, "FechaEmision")
    emisor_col = safe_col(chunk.columns, "RfcEmisor")
    receptor_col = safe_col(chunk.columns, "RfcReceptor")
    monto_col = safe_col(chunk.columns, "Monto")

    chunk[fecha_col] = pd.to_datetime(chunk[fecha_col], errors="coerce")
    chunk[monto_col] = pd.to_numeric(chunk[monto_col], errors="coerce")
    chunk.dropna(subset=[fecha_col], inplace=True)

    total_chunk = len(chunk)
    if total_chunk == 0:
        return 0, 0

    nuevos = ~chunk[uuid_col].isin(uuid_set)
    duplicados_chunk = (~nuevos).sum()

    chunk_unicos = chunk[nuevos].copy()
    uuid_set.update(chunk_unicos[uuid_col])

    if chunk_unicos.empty:
        return total_chunk, duplicados_chunk

    top_emisor = chunk_unicos[emisor_col].value_counts().idxmax()
    top_receptor = chunk_unicos[receptor_col].value_counts().idxmax()

    if chunk_unicos[emisor_col].value_counts().max() > chunk_unicos[receptor_col].value_counts().max():
        rfc_propio = top_emisor
        modo = "EMITIDAS"
    else:
        rfc_propio = top_receptor
        modo = "RECIBIDAS"

    print(f"🧠 RFC detectado: {rfc_propio} ({modo})")

    data = modos_data[modo]
    data["total_cfdi_cargados"] += total_chunk
    data["duplicados"] += duplicados_chunk
    data["total_cfdi_unicos"] += len(chunk_unicos)
    data["total_monto_unicos"] += chunk_unicos[monto_col].sum()

    # CFDI emitidos a sí mismo: ECE -> ECE
    ece_chunk = chunk_unicos[
        (chunk_unicos[emisor_col].astype(str) == ece_rfc) &
        (chunk_unicos[receptor_col].astype(str) == ece_rfc)
    ].copy()

    if not ece_chunk.empty:
        ece_chunk["Monto"] = pd.to_numeric(ece_chunk[monto_col], errors="coerce")
        ece_chunk["Mes"] = ece_chunk[fecha_col].dt.to_period("M").astype(str)
        ece_chunk["Modo"] = modo
        data["ece_rows"].append(ece_chunk.copy())
        data["ece_mensual"].append(ece_chunk.groupby("Mes").size())

    # análisis principal excluye autofacturación
    if modo == "EMITIDAS":
        chunk_unicos["RFC_ANALISIS"] = chunk_unicos[receptor_col]
    else:
        chunk_unicos["RFC_ANALISIS"] = chunk_unicos[emisor_col]

    chunk_analisis = chunk_unicos[chunk_unicos["RFC_ANALISIS"] != rfc_propio].copy()

    if chunk_analisis.empty:
        return total_chunk, duplicados_chunk

    chunk_analisis["Tipo"] = clasificar_rfc(chunk_analisis["RFC_ANALISIS"])
    chunk_analisis["Fecha"] = chunk_analisis[fecha_col].dt.date
    chunk_analisis["Mes"] = chunk_analisis[fecha_col].dt.to_period("M").astype(str)

    data["total_cfdi_analizados"] += len(chunk_analisis)
    data["total_monto_analizados"] += chunk_analisis[monto_col].sum()

    data["df_tipo_all"].append(chunk_analisis["Tipo"].value_counts())

    agg = chunk_analisis.groupby("RFC_ANALISIS").agg(
        CFDI=("Tipo", "count"),
        Monto=(monto_col, "sum"),
        Tipo=("Tipo", "first")
    )
    data["df_rfc_all"].append(agg)

    data["df_dia_all"].append(chunk_analisis.groupby("Fecha").size())
    data["df_mes_cfdi_all"].append(chunk_analisis.groupby("Mes").size())
    data["df_mes_monto_all"].append(chunk_analisis.groupby("Mes")[monto_col].sum())
    data["fechas_set"].update(chunk_analisis["Fecha"])

    mensual_proveedor = (
        chunk_analisis.groupby(["Mes", "RFC_ANALISIS"], as_index=False)
        .agg(
            CFDI=("Tipo", "count"),
            Monto=(monto_col, "sum"),
            Tipo=("Tipo", "first")
        )
    )
    data["mensual_proveedor_rows"].append(mensual_proveedor)

    gen_nac = chunk_analisis[chunk_analisis["Tipo"] == "Genérico Nacional"]
    gen_ext = chunk_analisis[chunk_analisis["Tipo"] == "Extranjero"]
    resto = chunk_analisis[~chunk_analisis["Tipo"].isin(["Genérico Nacional", "Extranjero"])]

    data["control_rows"].append({
        "Concepto": "Genérico Nacional",
        "CFDI": len(gen_nac),
        "Monto": gen_nac[monto_col].sum()
    })
    data["control_rows"].append({
        "Concepto": "Extranjero",
        "CFDI": len(gen_ext),
        "Monto": gen_ext[monto_col].sum()
    })
    data["control_rows"].append({
        "Concepto": "Resto de CFDI",
        "CFDI": len(resto),
        "Monto": resto[monto_col].sum()
    })

    return total_chunk, duplicados_chunk

def consolidar_modo_data(data, ece_rfc):
    tipo_final = (
        pd.concat(data["df_tipo_all"]).groupby(level=0).sum()
        if data["df_tipo_all"] else
        pd.Series(dtype="float64")
    )

    if data["df_rfc_all"]:
        proveedores_raw = pd.concat(data["df_rfc_all"]).reset_index()
        proveedores = (
            proveedores_raw.groupby("RFC_ANALISIS", as_index=False)
            .agg(
                CFDI=("CFDI", "sum"),
                Monto=("Monto", "sum"),
                Tipo=("Tipo", "first")
            )
            .sort_values(["CFDI", "Monto"], ascending=[False, False])
            .reset_index(drop=True)
        )
    else:
        proveedores = pd.DataFrame(columns=["RFC_ANALISIS", "CFDI", "Monto", "Tipo"])

    if data["df_dia_all"]:
        cfdi_dia = pd.concat(data["df_dia_all"]).groupby(level=0).sum().reset_index()
        cfdi_dia.columns = ["Fecha", "CFDI"]
        cfdi_dia = cfdi_dia.sort_values("Fecha").reset_index(drop=True)
    else:
        cfdi_dia = pd.DataFrame(columns=["Fecha", "CFDI"])

    if data["df_mes_cfdi_all"]:
        mensual = pd.concat(data["df_mes_cfdi_all"]).groupby(level=0).sum().reset_index()
        mensual.columns = ["Mes", "CFDI"]
        mensual = mensual.sort_values("Mes").reset_index(drop=True)
    else:
        mensual = pd.DataFrame(columns=["Mes", "CFDI"])

    if data["df_mes_monto_all"]:
        mensual_monto = pd.concat(data["df_mes_monto_all"]).groupby(level=0).sum().reset_index()
        mensual_monto.columns = ["Mes", "Monto"]
        mensual_monto = mensual_monto.sort_values("Mes").reset_index(drop=True)
    else:
        mensual_monto = pd.DataFrame(columns=["Mes", "Monto"])

    ece_detalle = pd.concat(data["ece_rows"], ignore_index=True) if data["ece_rows"] else pd.DataFrame()

    if data["ece_mensual"]:
        ece_mensual_final = pd.concat(data["ece_mensual"]).groupby(level=0).sum().reset_index()
        ece_mensual_final.columns = ["Mes", "CFDI"]
        ece_mensual_final = ece_mensual_final.sort_values("Mes").reset_index(drop=True)
    else:
        ece_mensual_final = pd.DataFrame(columns=["Mes", "CFDI"])

    total_cfdi_ece = len(ece_detalle)
    total_monto_ece = (
        pd.to_numeric(ece_detalle["Monto"], errors="coerce").sum()
        if not ece_detalle.empty and "Monto" in ece_detalle.columns else 0
    )

    ece_resumen = pd.DataFrame([{
        "RFC ECE": ece_rfc,
        "CFDI emitidos a sí mismo": total_cfdi_ece,
        "Monto emitido a sí mismo": total_monto_ece
    }])

    if data["fechas_set"]:
        rango = pd.date_range(min(data["fechas_set"]), max(data["fechas_set"]))
        faltantes = sorted(set(rango.date) - data["fechas_set"])
        if len(faltantes) == 0:
            validacion = pd.DataFrame({"Mensaje": ["NO FALTAN FECHAS"]})
        else:
            validacion = pd.DataFrame({"Fechas Faltantes": faltantes})
    else:
        validacion = pd.DataFrame({"Mensaje": ["SIN DATOS"]})

    top10 = proveedores.head(10).copy()

    control_base = pd.DataFrame([
        {"Concepto": "CFDI cargados", "CFDI": data["total_cfdi_cargados"], "Monto": np.nan},
        {"Concepto": "Duplicados", "CFDI": data["duplicados"], "Monto": np.nan},
        {"Concepto": "CFDI únicos", "CFDI": data["total_cfdi_unicos"], "Monto": data["total_monto_unicos"]},
        {
            "Concepto": f"CFDI emitidos a sí mismo ({ece_rfc}->{ece_rfc})",
            "CFDI": total_cfdi_ece,
            "Monto": total_monto_ece
        },
        {"Concepto": "CFDI analizados", "CFDI": data["total_cfdi_analizados"], "Monto": data["total_monto_analizados"]},
    ])

    if data["control_rows"]:
        control_detalle = pd.DataFrame(data["control_rows"]).groupby("Concepto", as_index=False).sum()
    else:
        control_detalle = pd.DataFrame(columns=["Concepto", "CFDI", "Monto"])

    control_final = pd.concat([control_base, control_detalle], ignore_index=True)

    orden_control = [
        "CFDI cargados",
        "Duplicados",
        "CFDI únicos",
        f"CFDI emitidos a sí mismo ({ece_rfc}->{ece_rfc})",
        "CFDI analizados",
        "Genérico Nacional",
        "Extranjero",
        "Resto de CFDI"
    ]

    control_final["Orden"] = control_final["Concepto"].apply(
        lambda x: orden_control.index(x) if x in orden_control else 999
    )
    control_final = control_final.sort_values(["Orden", "Concepto"]).drop(columns=["Orden"]).reset_index(drop=True)

    hojas_mensuales = {}
    if data["mensual_proveedor_rows"]:
        mensual_proveedor = pd.concat(data["mensual_proveedor_rows"], ignore_index=True)
        mensual_proveedor["Mes"] = mensual_proveedor["Mes"].astype(str)

        meses_detectados = sorted(mensual_proveedor["Mes"].dropna().unique())
        for mes in meses_detectados:
            tabla = (
                mensual_proveedor[mensual_proveedor["Mes"] == mes]
                .groupby("RFC_ANALISIS", as_index=False)
                .agg(
                    CFDI=("CFDI", "sum"),
                    Monto=("Monto", "sum"),
                    Tipo=("Tipo", "first")
                )
                .sort_values(["CFDI", "Monto"], ascending=[False, False])
                .reset_index(drop=True)
            )
            hojas_mensuales[mes] = tabla

    return {
        "tipo_final": tipo_final,
        "proveedores": proveedores,
        "cfdi_dia": cfdi_dia,
        "mensual": mensual,
        "mensual_monto": mensual_monto,
        "ece_detalle": ece_detalle,
        "ece_mensual_final": ece_mensual_final,
        "ece_resumen": ece_resumen,
        "validacion": validacion,
        "top10": top10,
        "control_final": control_final,
        "hojas_mensuales": hojas_mensuales
    }

def exportar_excel(nombre_archivo, modo, data, resultado, ece_rfc):
    with pd.ExcelWriter(nombre_archivo) as writer:
        pd.DataFrame([{
            "Modo": modo,
            "Total CFDI cargados": data["total_cfdi_cargados"],
            "Duplicados": data["duplicados"],
            "Total CFDI únicos": data["total_cfdi_unicos"],
            "Monto Total CFDI únicos": data["total_monto_unicos"],
            f"CFDI emitidos a sí mismo ({ece_rfc}->{ece_rfc})": int(
                resultado["control_final"].loc[
                    resultado["control_final"]["Concepto"] == f"CFDI emitidos a sí mismo ({ece_rfc}->{ece_rfc})",
                    "CFDI"
                ].sum()
            ),
            "Total CFDI analizados": data["total_cfdi_analizados"],
            "Monto Total CFDI analizados": data["total_monto_analizados"]
        }]).to_excel(writer, sheet_name="Resumen", index=False)

        resultado["control_final"].to_excel(writer, sheet_name="CUADRE_CONTROL", index=False)
        resultado["proveedores"].to_excel(writer, sheet_name="PROVEEDORES", index=False)
        resultado["top10"].to_excel(writer, sheet_name="TOP_10", index=False)
        resultado["mensual"].to_excel(writer, sheet_name="Mensual", index=False)
        resultado["mensual_monto"].to_excel(writer, sheet_name="MENSUAL_MONTO", index=False)
        resultado["cfdi_dia"].to_excel(writer, sheet_name="CFDI_DIA", index=False)
        resultado["validacion"].to_excel(writer, sheet_name="VALIDACION_FECHAS", index=False)
        resultado["ece_resumen"].to_excel(writer, sheet_name="ECE_RESUMEN", index=False)

        if not resultado["ece_mensual_final"].empty:
            resultado["ece_mensual_final"].to_excel(writer, sheet_name="ECE_MENSUAL", index=False)

        tipo_df = resultado["tipo_final"].reset_index()
        if not tipo_df.empty:
            tipo_df.columns = ["Tipo", "CFDI"]
            tipo_df.to_excel(writer, sheet_name="CFDI_POR_TIPO", index=False)

        for mes, tabla in resultado["hojas_mensuales"].items():
            nombre_hoja = sanitize_sheet_name(f"MES_{mes}")
            tabla.to_excel(writer, sheet_name=nombre_hoja, index=False)

# ============================================================
# 📂 CARGA
# ============================================================

archivos = []
uuid_set = set()
duplicados_globales = 0

while True:
    uploaded = files.upload()
    archivos.extend(uploaded.keys())
    campana()
    if input("¿Más archivos? (s/n): ").lower() != "s":
        break

print("\n🚀 Procesando...")

# ============================================================
# 🧠 ACUMULADORES
# ============================================================

ece_rfc = "ECE9610253TA"

modos_data = {
    "EMITIDAS": init_modo_data(),
    "RECIBIDAS": init_modo_data()
}

# ============================================================
# 🔍 PROCESAMIENTO
# ============================================================

for file in archivos:
    print(f"\n⚙️ {file}")
    archivo_procesado = False

    for sep in ["~", "|"]:
        try:
            reader = pd.read_csv(
                file,
                sep=sep,
                encoding="latin-1",
                engine="python",
                chunksize=100000,
                quoting=csv.QUOTE_NONE
            )

            first_chunk = next(reader)

            if len(first_chunk.columns) == 1:
                continue

            print(f"✅ Separador: {sep}")

            total_chunk, duplicados_chunk = procesar_chunk(first_chunk, uuid_set, ece_rfc, modos_data)
            duplicados_globales += duplicados_chunk

            for chunk in reader:
                total_chunk, duplicados_chunk = procesar_chunk(chunk, uuid_set, ece_rfc, modos_data)
                duplicados_globales += duplicados_chunk

            archivo_procesado = True
            break

        except StopIteration:
            print(f"⚠️ Archivo vacío o sin datos legibles: {file}")
            break

        except Exception as e:
            print(f"⚠️ Falló con separador '{sep}' en {file}: {e}")
            continue

    if not archivo_procesado:
        print(f"❌ No se pudo procesar correctamente: {file}")

# ============================================================
# 🧪 DEBUG
# ============================================================

print("\n🧪 DEBUG:")
for modo in ["EMITIDAS", "RECIBIDAS"]:
    print(
        modo,
        "df_tipo_all:", len(modos_data[modo]["df_tipo_all"]),
        "df_rfc_all:", len(modos_data[modo]["df_rfc_all"])
    )

# ============================================================
# 📊 CONSOLIDACIÓN
# ============================================================

resultados = {}
for modo in ["EMITIDAS", "RECIBIDAS"]:
    resultados[modo] = consolidar_modo_data(modos_data[modo], ece_rfc)

# ============================================================
# 📊 KPI CONSOLA
# ============================================================

print("\n================ KPI =================")
print(f"📂 Archivos cargados: {len(archivos)}")
print(f"🆔 UUID globales únicos: {len(uuid_set):,}")
print(f"🧹 Duplicados globales: {duplicados_globales:,}")

for modo in ["EMITIDAS", "RECIBIDAS"]:
    data = modos_data[modo]
    resultado = resultados[modo]

    if data["total_cfdi_cargados"] > 0:
        print(f"\n================ {modo} ================")
        print(f"📥 CFDI cargados: {data['total_cfdi_cargados']:,}")
        print(f"🧹 Duplicados: {data['duplicados']:,}")
        print(f"✅ CFDI únicos: {data['total_cfdi_unicos']:,}")
        print(f"💰 Monto CFDI únicos: {data['total_monto_unicos']:,.2f}")

        cfdi_ece = int(
            resultado["control_final"].loc[
                resultado["control_final"]["Concepto"] == f"CFDI emitidos a sí mismo ({ece_rfc}->{ece_rfc})",
                "CFDI"
            ].sum()
        )
        monto_ece = resultado["control_final"].loc[
            resultado["control_final"]["Concepto"] == f"CFDI emitidos a sí mismo ({ece_rfc}->{ece_rfc})",
            "Monto"
        ].sum()

        print(f"🔁 CFDI emitidos a sí mismo: {cfdi_ece:,}")
        print(f"💰 Monto emitido a sí mismo: {monto_ece:,.2f}")
        print(f"🧾 CFDI analizados: {data['total_cfdi_analizados']:,}")
        print(f"💰 Monto CFDI analizados: {data['total_monto_analizados']:,.2f}")

        print("\n--- CFDI POR TIPO ---")
        if not resultado["tipo_final"].empty:
            print(resultado["tipo_final"])
        else:
            print("SIN DATOS")

        print("\n--- TOP 10 PROVEEDORES ---")
        if not resultado["top10"].empty:
            print(resultado["top10"])
        else:
            print("SIN DATOS")

        if not resultado["ece_detalle"].empty:
            print(f"\n📦 ECE_DETALLE ({modo}): {len(resultado['ece_detalle']):,}")

# ============================================================
# 📁 EXPORTACIÓN
# ============================================================

archivos_salida = []

for modo in ["EMITIDAS", "RECIBIDAS"]:
    data = modos_data[modo]
    resultado = resultados[modo]

    if data["total_cfdi_cargados"] > 0:
        nombre_excel = f"KPI_ENGINE_{modo}.xlsx"
        exportar_excel(nombre_excel, modo, data, resultado, ece_rfc)
        archivos_salida.append(nombre_excel)
        print(f"\n✅ Archivo generado: {nombre_excel}")

    if not resultado["ece_detalle"].empty:
        nombre_csv = f"ECE_DETALLE_{modo}.csv"
        resultado["ece_detalle"].to_csv(nombre_csv, index=False, encoding="utf-8-sig")
        archivos_salida.append(nombre_csv)
        print(f"✅ CSV generado: {nombre_csv}")

# ============================================================
# ⬇️ DESCARGA
# ============================================================

for archivo in archivos_salida:
    files.download(archivo)


Saving 1E4122DD-A54B-4BA2-AF32-6CC5E01EEB93_01.txt to 1E4122DD-A54B-4BA2-AF32-6CC5E01EEB93_01 (3).txt
Saving 05FAF6DC-DD27-4900-B111-FA453243869A_01.txt to 05FAF6DC-DD27-4900-B111-FA453243869A_01 (3).txt


¿Más archivos? (s/n): n

🚀 Procesando...

⚙️ 1E4122DD-A54B-4BA2-AF32-6CC5E01EEB93_01 (3).txt
✅ Separador: ~
🧠 RFC detectado: ECE9610253TA (EMITIDAS)
🧠 RFC detectado: ECE9610253TA (EMITIDAS)

⚙️ 05FAF6DC-DD27-4900-B111-FA453243869A_01 (3).txt
✅ Separador: ~
🧠 RFC detectado: ECE9610253TA (EMITIDAS)
🧠 RFC detectado: ECE9610253TA (EMITIDAS)

🧪 DEBUG:
EMITIDAS df_tipo_all: 4 df_rfc_all: 4
RECIBIDAS df_tipo_all: 0 df_rfc_all: 0

================ KPI =================
📂 Archivos cargados: 2
🆔 UUID globales únicos: 290,830
🧹 Duplicados globales: 0

================ EMITIDAS ================
📥 CFDI cargados: 290,830
🧹 Duplicados: 0
✅ CFDI únicos: 290,830
💰 Monto CFDI únicos: 1,681,890,853.90
🔁 CFDI emitidos a sí mismo: 613
💰 Monto emitido a sí mismo: 2,199.00
🧾 CFDI analizados: 290,217
💰 Monto CFDI analizados: 1,681,888,654.90

--- CFDI POR TIPO ---
Tipo
Extranjero               59
Genérico Nacional    270825
Persona Física        16080
Persona Moral          3253
Name: count, dtype: int64

---

/tmp/ipykernel_2242/6275244.py:292: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  control_final = pd.concat([control_base, control_detalle], ignore_index=True)



✅ Archivo generado: KPI_ENGINE_EMITIDAS.xlsx
✅ CSV generado: ECE_DETALLE_EMITIDAS.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>